 # 智能体
### 1.4.2 全新的调用

In [27]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式

)



In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


# 2. 定义工具（示例）
@tool
def tool1(query: str) -> str:
      """示例工具1"""
      return f"结果: {query}"


@tool
def tool2(query: str) -> str:
      """示例工具2"""
      return f"结果: {query}"


# 1. 初始化模型
model = init_chat_model("deepseek-v4-flash", model_provider="deepseek")

# 2. 创建 agent （一步完成）
agent = create_agent(
      model=model, tools=[tool1, tool2], system_prompt="Agent 的行为指令 "  # 可选
)

# 3. 调用
result = agent.invoke({"messages": [{"role": "user", "content": " 问题 "}]})

##  Agent 的基本用法 1 ：模型的传入方式

In [ ]:
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model: str | BaseChatModel, # 必需：聊天模型
#     tools: List[BaseTool], # 必需：工具列表
#     *,
#     system_prompt: str = "", # 系统提示词
#     middleware: Seguence[AgentMiddleware[StateT_co, ContextT]] = () # 中间件
#     interrupt_before: List[str] = None, # 在某些工具前暂停（人机协作）
#     interrupt_after: List[str] = None, # 在某些工具后暂停
#     debug: bool = False # 调试模式
#     name: str 丨 None = None, # 设置模型名称
# )

## 2.1 传入模型字符串

In [ ]:
from langchain.agents import create_agent

from dotenv import load_dotenv

load_dotenv(override=True)

agent = create_agent("deepseek-v4-flash")
print(type(agent))

from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

### 2.2 传入模型对象

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式

)

agent = create_agent(model=model)

response = agent.invoke({"messages": [" 你好 "]})  # 默认是 HumanMessage
print(type(response))
rprint(response)

### **举例 2：**
invoke 调用的核心就是输入一系列消息（ messages ），每条消息通常包含 role （如 "user", "assistant", "system", "tool" ）和 "content" 。
我们也可以在 message 列表的开头加入 "system" 角色的消息来定义 Agent 的行为。

In [ ]:
agent = create_agent(model)

resp = agent.invoke(
      {
            "messages": [
                  {
                        "role": "system",
                        "content": " 你是一个小学数学老师，耐心，幽默，讲解深入浅出 ",
                  },
                  {"role": "user", "content": "100 加上 50 等于多少？ "},
            ]
      }
)

rprint(resp)

### ## 4、 Agent 的基本用法 3 ：绑定工具  Agents 支持绑定一或多个工具。

只有接入了一些工具， create_agent 完成 Agent 创建才算完整。
Agent 支持 静态 和 动态 绑定工具，后者需要用到 中间件

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      天气查询工具

      Args:
      city: 城市名称
      """
      return f"{city} 的天气为晴朗， 25°C 。 "


agent = create_agent(model=model, tools=[get_weather])

resp = agent.invoke(
      {
            "messages": [
                  {
                        "role": "system",
                        "content": " 你是一个天气查询助手，只回答天气相关的问题，其他问题请直接回答：我不清楚这问题答案。 ",
                  },
                  {"role": "user", "content": " 北京的天气怎么样？ "},
                  # {"role": "user", "content": "100 加上 50 等于多少？ "}
            ]
      }
)

rprint(resp)

### 举例 2 ：接入内置工具

In [16]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os

load_dotenv(override=True)

web_search = TavilySearch(tavily_api_key=os.getenv("TAVILY_API_KEY"), max_results=2, )

# 这是一个高度封装的网络搜索工具，可以直接调用：
result = web_search.invoke(" 请问 2026 年足球世界杯有哪些参赛队？ ")
{'query': ' 请问 2026 年足球世界杯有哪些参赛队？ ',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.instagram.com/p/DWk8mU-geNg',
              'title': ' 你準備好了嗎？ 2026 FIFA 世界盃 ™ 48 支隊伍全數到齊 ... - Instagram',
              'content': ' 本屆賽事由美國、加拿大、墨西哥三國聯合主辦，總共 48 支球隊、 104 場比賽、 39天賽程，這次的世界盃賽程本身就是前所未見的規模',
              'score': 0.9974885,
              'raw_content': None},
             {'url': 'https://zh.wikipedia.org/zh-'}]}
rprint(result)

{
    'query': '请问 2026 年足球世界杯有哪些参赛队？',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'url': 
'https://www.aps.dz/cn/sports/sport-collectif/mnfzpl39-2026%E5%B9%B4%E4%B8%96%E7%95%8C%E6%9D%AF48%E6%94%AF%E5%8F%82
%E8%B5%9B%E5%9B%BD%E5%AE%B6',
            'title': '2026年世界杯48支参赛国家 | 阿尔及利亚新闻社',
            'content': 'Algeria\n\nAPS 在线\n\nAPS 视频\n\nAPS 图片\n\n档案\n\n中文\n\ntwitter\n\n# 
2026年世界杯48支参赛国家\n\n2026年四月1日 周三 
12:56\n\n2026年世界杯48支参赛国家\n\n巴黎（法国）——伊拉克于周三获得了2026年世界杯的最后一个参赛资格（6月11日—7月19
日），这是首次将参赛队伍扩充至48支，并由三个国家（美国、加拿大和墨西哥）联合举办的世界杯。\n\n相关文章\n\n2026年世
界杯48支参赛国家 | 阿尔及利亚新闻社',
            'score': 0.7364006,
            'raw_content': None,
            'id': 'fbc05f-00'
        },
        {
            'url': 
'https://baike.baidu.com/item/2026%E5%B9%B4%E5%9B%BD%E9%99%85%E8%B6%B3%E8%81%94%E4%B8%96%E7%95%8C%E6%9D%AF/62863018
',
            'title': '2026年国际足联世界杯',
            'content': 'I组Image 82Image 83Image 84Image 85\n🇫🇷 法国🇸🇳 塞内加尔🇮🇶 伊拉克🇳🇴 挪威\nJ组Image 86Image 
87Image 88Image 89\n🇦🇷 阿根廷🇩🇿 阿尔及利亚🇦🇹 奥地利🇯🇴 约旦\nK组Image 90Image 91Image 92Image 93\n🇵🇹 葡萄牙🇨🇩 
刚果（金）🇺🇿 乌兹别克斯坦🇨🇴 哥伦比亚\nL组Image 94Image 95Image 96Image 
97\n🏴\U000e0067\U000e0062\U000e0065\U000e006e\U000e0067\U000e007f 英格兰🇭🇷 克罗地亚🇬🇭 加纳🇵🇦 巴拿马\n参考资料： 
[...] ## 赛事阵容\n\n播报\n\n编辑\n\n本届世界杯1248名参赛球员分属全球71个国家和地区的449家职业俱乐部。 \n\n### 
参赛球队\n\n2026年4月1日，伊拉克队在2026国际足联世界杯预选赛洲际附加赛决赛中2:1击败玻利维亚队，获得该届世界杯最后一
个参赛席位。至此该届世界杯48支参赛球队全部产生。其中，亚足联共9支球队参赛，创历史新高。 [...] 
作为联合东道主的加拿大、墨西哥和美国三支队伍都被分入第一档，39支已晋级球队根据19日发布的世界排名分配到四个档次。两
支通过国际足联附加赛晋级的球队以及四支通过欧洲区附加赛出线的队伍被分入第四档。\n\n球队分档情况如下：\n\n第一档：加
拿大、墨西哥、美国、西班牙、阿根廷、法国、英格兰、巴西、葡萄牙、荷兰、比利时、德国；\n\n第二档：克罗地亚、摩洛哥、
哥伦比亚、乌拉圭、瑞士、日本、塞内加尔、伊朗、韩国、厄瓜多尔、奥地利、澳大利亚；\n\n第三档：挪威、巴拿马、埃及、阿
尔及利亚、苏格兰、巴拉圭、突尼斯、科特迪瓦、乌兹别克斯坦、卡塔尔、沙特阿拉伯、南非；\n\n第四档：约旦、佛得角、加纳
、库拉索、海地、新西兰、欧洲区附加赛四队、国际足联附加赛两队 。\n\n   分组结果',
            'score': 0.72535694,
            'raw_content': None,
            'id': '9906a4-01'
        }
    ],
    'response_time': 0.0,
    'request_id': 'da877b03-31f9-4e96-a8f6-52b32d5c1a2f'
}

可以直接带入 create_agent 中作为外部工具。

In [19]:
# 2. 工具实例化
web_search = TavilySearch(max_results=2)
# 3. 创建 Agent
agent = create_agent(
      model=model,
      tools=[web_search],
      # system_prompt=" 你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。 "
)

# 4. 运行 Agent 获得结果
result = agent.invoke(
      {
            "messages": [
                  {"role": "user", "content": " 请帮我查询 2024 年诺贝尔物理学奖得主是谁？ "}
            ]
      }
)

# rprint(result)
print(result['messages'][-1].content)

2024 年诺贝尔物理学奖的得主是：

**约翰·霍普菲尔德（John J. Hopfield）** 和 **杰弗里·辛顿（Geoffrey E. Hinton）**

获奖理由：表彰他们**"基于人工神经网络的机器学习基础性发现与发明"**（"for foundational discoveries and inventions that enable machine learning with artificial neural networks"）。

- **约翰·霍普菲尔德**（美国普林斯顿大学）——因发明了能够存储和重建信息的**霍普菲尔德网络**（Hopfield network）而获奖。
- **杰弗里·辛顿**（加拿大多伦多大学）——因发明了能够自主发现数据特征的**玻尔兹曼机**（Boltzmann machine）而获奖，他是利用统计物理学工具训练人工神经网络方法的先驱。

两人分享了 1100 万瑞典克朗的奖金。颁奖时间为 2024 年 10 月 8 日公布，正式颁奖典礼于 12 月 10 日在斯德哥尔摩举行。

资料来源：诺贝尔奖官方网站（nobelprize.org）。


### 举例 3 ：绑定多个工具

In [ ]:
load_dotenv()


@tool(parse_docstring=True)
def get_weather(city: str):
      """
      天气查询工具

      Args:
      city: 城市名称
      """
      return f"{city} 今天天气挺好 "


@tool(parse_docstring=True)
def get_news():
      """
      新闻查询工具
      """
      return " 近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来 “ 火热潮 ” ，回收价格普遍上涨，旧手机成 “ 香饽饽 ” 。 "


agent = create_agent(model, tools=[get_weather, get_news])
response = agent.invoke({"messages": [" 你好，杭州今天的天气如何？今天有哪些新闻？ "]})

rprint(response)

## 4.3 重试机制
Agent 可以在工具调用结果不满足要求时，自主重试。

In [21]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)

flag = 0


@tool
def get_weather(city: str):
      """
      天气查询工具

      Args:
      city: 城市名称
      """
      global flag
      flag += 1

      if flag < 3:
            # raise Exception(" 暂时无法访问 ")
            return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试 "

      return f"{city} 今天天气挺好 "


messages = [
      SystemMessage(
            """
        你是一个天气助手。
        当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，
        说明是临时故障，不要立即放弃；
        你应再次调用同一个工具，最多重试 3 次。
        如果 3 次后仍失败，再向用户说明服务暂时不可用。
        """
      ),
      HumanMessage(" 你好，杭州今天的天气如何？ "),
]
agent = create_agent(model, tools=[get_weather])
response = agent.invoke({"messages": messages})

# rprint(response)

for msg in response["messages"]:
      msg.pretty_print()

================================ System Message ================================


    你是一个天气助手。
    当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，
    说明是临时故障，不要立即放弃；
    你应再次调用同一个工具，最多重试 3 次。
    如果 3 次后仍失败，再向用户说明服务暂时不可用。
    
================================ Human Message =================================

 你好，杭州今天的天气如何？ 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_rNdBHJmPUNVt3DszK4AW4789)
 Call ID: call_00_rNdBHJmPUNVt3DszK4AW4789
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_mRJTvo6wt1YDQBHyhLPz6153)
 Call ID: call_00_mRJTvo6wt1YDQBHyhLPz6153
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试 
===

## 6、 Agent 的高级用法

In [34]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# 2. 导入工具
web_search = TavilySearch(max_results=2)

# 3. 创建 Agent
agent = create_agent(
      model=model,
      tools=[web_search],
      system_prompt=" 你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。 ",
)

# 4. 运行 Agent 获得结果
result = agent.invoke(
      {
            "messages": [
                  {
                        "role": "user",
                        "content": " 请帮我查询 2026 年足球世界杯是哪个国家举办的？ ",
                  }
            ]
      }
)

print(result['messages'][-1].content)

2026年足球世界杯（FIFA World Cup 2026）将由**美国、加拿大、墨西哥**三国联合举办。

这将是历史上首次由三个国家共同承办的世界杯，也是自2002年韩日世界杯以来再次出现多国联合主办的情况。赛事计划于2026年6月至7月举行，共有48支球队参赛（首次扩军至48队），比赛将在北美16个城市的场馆进行。

需要我帮你查询具体的举办城市、赛程或门票信息吗？


**举例 2：**

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint


# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
      """计算并返回两个数的和。"""
      return f" 和为： {a + b}"


# 创建客服助手 Agent
agent = create_agent(
      model=model,
      tools=[add_numbers],  # 工具列表
      # system_prompt=" 你是一个数学助手，解决日常的算术问题 "
      system_prompt=SystemMessage(content=" 你是一个数学助手，解决日常的算术问题 "),
)

response = agent.invoke(
      {"messages": [{"role": "user", "content": "10 加上 20 再加上 30 是多少？ "}]},
)

rprint(response)
# print(response["messages"][-1].content)

 **举例 3：**

In [24]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0


@tool
def get_weather(city: str):
      """
      天气查询工具

      Args:
      city: 城市名称
      """
      global flag
      flag += 1

      if flag < 3:
            # raise Exception(" 暂时无法访问 ")

            return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试 "

      return f"{city} 今天天气挺好 "


messages = [HumanMessage(" 你好，杭州今天的天气如何？ ")]

agent = create_agent(
      model=model,
      tools=[get_weather],
      system_prompt=SystemMessage(
            " 你是一个天气助手。 "
            " 当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时， "
            " 说明是临时故障，不要立即放弃； "
            " 你应再次调用同一个工具，最多重试 3 次。 "
            " 如果 3 次后仍失败，再向用户说明服务暂时不可用。 "
      ),
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
      msg.pretty_print()

================================ Human Message =================================

 你好，杭州今天的天气如何？ 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_iTzIXBAw8jl6IM2BZ1g05570)
 Call ID: call_00_iTzIXBAw8jl6IM2BZ1g05570
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试 
================================== Ai Message ==================================

我第一次查询遇到了临时故障，我再试一次。
Tool Calls:
  get_weather (call_00_Y7FnxcvEEf6K1hMGdc4O4383)
 Call ID: call_00_Y7FnxcvEEf6K1hMGdc4O4383
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试 
================================== Ai Message ==================================

第二次仍然失败，我再尝试一次。
Tool Calls:
  get_weather (call_00_an4tsev5WOA2rX3rpEpZ4825)
 Call ID: call_00_an4tsev5WOA2rX3rpEpZ4

## 7.2 结构化输出的 4 种策略
ProviderStrategy      适用于支持原生结构化输出的模型，比如 OpenAI 、 Anthropic Claude 或 xAI Grok 等。

In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage


# 2.Pydantic 结构化方式定义
class ContactInfo(BaseModel):
      """用户的联系方式"""

      name: str = Field(description=" 用户姓名 ")
      email: str = Field(description=" 用户邮箱地址 ")
      phone: str = Field(description=" 用户的手机号 ")


# 3.agent 初始化
agent = create_agent(model=model, response_format=ProviderStrategy(ContactInfo))

# 4. 调用
response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

# rprint(response)

for msg in response["messages"]:
      msg.pretty_print()

OpenAIInvalidRequestError: Error code: 400 - This response_format type is unavailable now (request_id: bafe0daf-eef0-4530-aee0-0c1c1f9a5dbf)

### ToolStrategy

In [31]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

phone: str = Field(description=" 电话 ")


# 3. 工具的定义（根据需要定义）
@tool
def search_tool(query: str) -> str:
      """这是一个搜索引擎。当大模型发现给定的上下文里缺少必要的联系人信息，
      需要去互联网上查询时，才会调用这个工具。
      """
      return f" 搜索结果 : 未找到关于 '{query}' 的更多额外信息。 "


# 3.agent 初始化
agent = create_agent(
      model=model, tools=[search_tool], response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke(
      {
            "messages": [
                  {
                        "role": "user",
                        "content": " 联系人信息 : John Doe,john@atguigu.com, (010) 56253825",
                  }
            ]
      }
)

# print(result)
print(result["structured_response"])

name='John Doe' email='john@atguigu.com' phone='(010) 56253825'


### ContactInfo

In [33]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
      """联系人信息"""

      name: str = Field(description=" 姓名 ")
      email: str = Field(description=" 邮箱 ")
      phone: str = Field(description=" 电话 ")


agent = create_agent(
      model=model, response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
      {
            "messages": [
                  {
                        "role": "user",
                        "content": " 联系人信息 : John Doe,john@atguigu.com, (010) 56253825",
                  }
            ]
      }
)

# print(result)
print(result["structured_response"])

OpenAIInvalidRequestError: Error code: 400 - This response_format type is unavailable now (request_id: 9ec5db8b-0406-400a-a018-eb181bcbb27c)

### ToolStrategy 使用详解

In [36]:
"""
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]


schema （必需参数） ：与提供商策略的 schema 参数功能一致，支持 Pydantic 模型 、 TypedDict 、 JSON Schema 、 数据类 (@dataclass)
，同时还支持联合类型 Union[ 类型 1, 类型2 （允许模型根据输入内容选择最匹配的数据结构）。

tool_message_content （可选参数） ：用于自定义生成结构化输出时，会话历史中记录的提示信息。默认使用展示输出数据的标准响应语句。

handle_errors （可选参数） ：用于指定数据校验失败时的重试策略， 默认值为 True 。
"""


OpenAIInvalidRequestError: Error code: 400 - This response_format type is unavailable now (request_id: ab965ca7-8695-4e99-8770-0d0c4e76f84a)